In [3]:
import pandas as pd

In [4]:
import numpy as np

In [5]:
df = pd.read_csv("/Users/mac/Desktop/new-bref/darkom-annonces-6a0a532a16460470060059.csv")

In [6]:
df.shape

(1508, 13)

In [7]:
df.columns

Index(['annonce_id', 'date_publication', 'titre', 'ville', 'quartier',
       'type_bien', 'transaction', 'prix', 'surface', 'nb_chambres',
       'nb_salles_bain', 'etage', 'annee_construction'],
      dtype='str')

In [9]:
df.dtypes

annonce_id                str
date_publication          str
titre                     str
ville                     str
quartier                  str
type_bien                 str
transaction               str
prix                  float64
surface               float64
nb_chambres           float64
nb_salles_bain        float64
etage                 float64
annee_construction    float64
dtype: object

In [10]:
df.head()

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction
0,ANO000343,2023-05-05,Terrain moderne Meknès,Meknès,Hamria,Terrain,Vente,811401.04,314.40,0.0,NaN,0.0,2014.0
1,ANO001296,2023-08-20,Terrain moderne Kenitra,Kenitra,Centre,Terrain,Location,3843.55,36.46,0.0,0.0,NaN,2008.0
2,ANO001212,2023-06-20,Appartement moderne Oujda,Oujda,Hay Qods,Appartement,Location,9190.70,95.54,2.0,2.0,NaN,2012.0
3,ANO001000,2024-01-21,Appartement à vente - Tanger,Tanger,NaN,Appartement,Vente,30000.00,93.35,1.0,1.0,3.0,2008.0
4,ANO001123,2024-03-19,Beau Villa Rabat,Rabat,Souissi,Villa,Location,20844.72,264.22,3.0,1.0,0.0,2006.0


In [15]:
df.duplicated().sum()

np.int64(8)

In [16]:
df.drop_duplicates(keep='first', inplace=True)

In [18]:
df.isnull().sum()

annonce_id              0
date_publication       76
titre                   0
ville                   0
quartier              415
type_bien              38
transaction            38
prix                    0
surface                 0
nb_chambres           129
nb_salles_bain        103
etage                 232
annee_construction    203
dtype: int64

In [19]:
df["date_publication"] = pd.to_datetime(df["date_publication"])

In [20]:
df["date_publication"] = df["date_publication"].ffill().bfill()

In [30]:
df["date_publication"].sample(10)

347    2024-04-04
1276   2024-02-26
965    2024-09-24
637    2023-02-16
253    2023-10-18
1260   2024-03-10
1442   2023-03-16
1149   2024-12-08
500    2024-04-15
1281   2023-11-03
Name: date_publication, dtype: datetime64[us]

In [31]:
df["date_publication"].isnull

<bound method Series.isnull of 0      2023-05-05
1      2023-08-20
2      2023-06-20
3      2024-01-21
4      2024-03-19
          ...    
1503   2024-10-22
1504   2024-01-05
1505   2023-08-30
1506   2024-06-05
1507   2024-02-23
Name: date_publication, Length: 1500, dtype: datetime64[us]>

In [32]:
df["ville"].value_counts()

ville
Marrakech     147
Tétouan       144
Meknès        142
Casablanca    140
Kenitra       137
Tanger        135
Rabat         131
Oujda         130
Agadir        130
Fès           119
tétouan        13
MARRAKECH      12
TANGER         11
oujda           9
TÉTOUAN         9
MEKNÈS          8
rabat           8
RABAT           7
CASABLANCA      7
OUJDA           7
casablanca      7
fès             6
FÈS             6
Casa            6
tanger          5
agadir          5
kenitra         5
KENITRA         4
meknès          4
marrakech       3
AGADIR          3
Name: count, dtype: int64

In [33]:
df["ville"] = df["ville"].str.lower().str.strip()

In [34]:
import unicodedata

def remove_accents(text):
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

df["ville"] = df["ville"].apply(remove_accents)

In [35]:
df["ville"] = df["ville"].replace({
    "casa": "casablanca",
    "casablanca": "casablanca",

    "marrakech": "marrakech",
    "meknes": "meknes",
    "tanger": "tanger",
    "tetouan": "tetouan",
    "rabat": "rabat",
    "oujda": "oujda",
    "fes": "fes",
    "agadir": "agadir",
    "kenitra": "kenitra"
})

In [36]:
df["ville"] = df["ville"].replace(
    to_replace=r".*casa.*",
    value="casablanca",
    regex=True
)

In [37]:
df["ville"].value_counts()

ville
tetouan       166
marrakech     162
casablanca    160
meknes        154
tanger        151
kenitra       146
oujda         146
rabat         146
agadir        138
fes           131
Name: count, dtype: int64

In [38]:
df["quartier"].isnull().sum()

np.int64(415)

In [40]:
df["quartier"].isnull().value_counts(normalize=True)

quartier
False    0.723333
True     0.276667
Name: proportion, dtype: float64

In [41]:
df["quartier"].value_counts()

quartier
Centre            168
Ville Nouvelle     83
Medina             74
Hay Qods           58
M'diq              58
Hamria             51
Hay Mohammadi      48
Hay Essalam        45
Talborjt           44
Iberia             35
Centre Ville       34
Malabata           32
Targa              31
Hay Riad           30
Zouagha            28
Hivernage          28
Gueliz             27
Ocean              27
Hassan             26
Bourgogne          25
Souissi            23
Maarif             23
Ain Diab           20
Californie         19
Anfa               19
Non spécifié       15
Agdal              14
Name: count, dtype: int64

In [42]:
ville_quartier_map = df.groupby("ville")["quartier"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else "unknown"
)

In [43]:
ville_quartier_map = ville_quartier_map.to_dict()

In [44]:
df["quartier"] = df["quartier"].fillna(df["ville"].map(ville_quartier_map))

In [45]:
df["quartier"].isnull().sum()

np.int64(0)

In [48]:
df["quartier"].sample(10)

1004          Hay Qods
1023    Ville Nouvelle
482               Anfa
1096          Hay Qods
1461          Hay Qods
1262    Ville Nouvelle
738           Hay Qods
1117             M'diq
735             Centre
421              Targa
Name: quartier, dtype: str

In [49]:
df["type_bien"].isnull().sum()

np.int64(38)

In [50]:
df["type_bien"] = df["type_bien"].str.lower().str.strip()

In [52]:
types = ["villa", "appartement", "terrain", "duplex", "bureau"]

for t in types:
    mask = df["type_bien"].isnull() & df["titre"].str.lower().str.contains(t)
    df.loc[mask, "type_bien"] = t

In [53]:
df["type_bien"].isnull().sum()

np.int64(0)

In [54]:
df["type_bien"] = df["type_bien"].astype("category")

In [55]:
df["type_bien"].value_counts()

type_bien
appartement    766
villa          370
terrain        151
bureau         144
duplex          69
Name: count, dtype: int64

In [56]:
df["transaction"].isnull().sum()

np.int64(38)

In [57]:
df["prix"].describe()

count    1.500000e+03
mean     1.732155e+06
std      6.251217e+06
min      1.200000e+02
25%      1.154916e+04
50%      6.057739e+05
75%      1.821021e+06
max      8.000000e+07
Name: prix, dtype: float64

In [58]:
low = df["prix"].quantile(0.4)
high = df["prix"].quantile(0.6)

In [60]:
df["transaction"] = np.where(
    df["prix"] <= low, "location",
    np.where(df["prix"] >= high, "vente", None)
)

In [61]:
df["transaction"] = df["transaction"].fillna(df["transaction"].mode()[0])

In [62]:
df["transaction"] = df["transaction"].astype("category")

In [63]:
df["transaction"].isnull().sum()

np.int64(0)

In [66]:
df["transaction"].value_counts()

transaction
location    900
vente       600
Name: count, dtype: int64

In [67]:
df.dtypes

annonce_id                       str
date_publication      datetime64[us]
titre                            str
ville                            str
quartier                         str
type_bien                   category
transaction                 category
prix                         float64
surface                      float64
nb_chambres                  float64
nb_salles_bain               float64
etage                        float64
annee_construction           float64
dtype: object

In [68]:
df.isnull().sum()

annonce_id              0
date_publication        0
titre                   0
ville                   0
quartier                0
type_bien               0
transaction             0
prix                    0
surface                 0
nb_chambres           129
nb_salles_bain        103
etage                 232
annee_construction    203
dtype: int64

In [70]:
df["nb_chambres"].isnull().sum()

np.int64(129)

In [ ]:
median_chambrs = df.groupby(["ville", "type_bien"])["nb_chambres"].transform("median")

In [ ]:
df["nb_chambres"] = df["nb_chambres"].fillna(median_chambrs)

In [75]:
df["nb_chambres"].value_counts()

nb_chambres
2.0     354
3.0     320
0.0     292
4.0     246
1.0     115
5.0     111
6.0      38
2.5       8
25.0      7
18.0      5
20.0      4
Name: count, dtype: int64

In [76]:
df["nb_chambres"] = df["nb_chambres"].astype(int)

In [77]:
df["nb_chambres"].value_counts()

nb_chambres
2     362
3     320
0     292
4     246
1     115
5     111
6      38
25      7
18      5
20      4
Name: count, dtype: int64

In [80]:
df["nb_salles_bain"].isnull().sum()

np.int64(103)

In [ ]:
median_bain = df.groupby(["ville", "type_bien"])["nb_salles_bain"].transform("median")

In [ ]:
df["nb_salles_bain"] = df["nb_salles_bain"].fillna(median_bain)

In [89]:
df["nb_salles_bain"].value_counts()

nb_salles_bain
2.0     514
1.0     471
0.0     295
3.0     185
4.0      21
13.0      5
10.0      5
9.0       2
11.0      1
2.5       1
Name: count, dtype: int64

In [90]:
df["nb_salles_bain"].isnull().sum()

np.int64(0)

In [91]:
df["nb_chambres"] = df["nb_chambres"].astype(int)

In [92]:
df["etage"].isnull().sum()

np.int64(232)

In [ ]:
median_etage = df.groupby(["ville", "type_bien"])["etage"].transform("median")

In [ ]:
df["etage"] = df["etage"].fillna(median_etage)

In [95]:
df["etage"].isnull().sum()

np.int64(0)

In [96]:
df["etage"] = df["etage"].astype(int)

In [98]:
df["annee_construction"].isnull().sum()

np.int64(203)

In [99]:
mode_annee_construction = df.groupby(["ville", "type_bien"])["annee_construction"].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)

In [100]:
df["annee_construction"] = df["annee_construction"].fillna(mode_annee_construction)

In [101]:
df["annee_construction"].isnull().sum()

np.int64(0)

In [102]:
df["annee_construction"] = df["annee_construction"].astype(int)

In [103]:
df.dtypes

annonce_id                       str
date_publication      datetime64[us]
titre                            str
ville                            str
quartier                         str
type_bien                   category
transaction                 category
prix                         float64
surface                      float64
nb_chambres                    int64
nb_salles_bain               float64
etage                          int64
annee_construction             int64
dtype: object

In [104]:
df.sample(n=15,random_state=42)

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction
1121,ANO001365,2023-03-21,Beau Villa Meknès,meknes,Hamria,villa,vente,5829779.03,267.72,3,3.0,0,2019
1376,ANO000913,2023-08-05,Appartement moderne Fès,fes,Ville Nouvelle,appartement,location,7463.78,93.56,1,1.0,1,2023
422,ANO001425,2023-04-12,Appartement moderne Tanger,tanger,Iberia,appartement,location,386971.12,58.01,1,1.0,9,1993
413,ANO000967,2023-02-23,Appartement 92m² Centre,kenitra,Centre,appartement,location,515506.30,92.18,1,1.0,1,2009
451,ANO001329,2024-10-15,Appartement moderne Agadir,agadir,Centre,appartement,location,579145.84,32.95,2,2.0,3,2014
863,ANO000801,2024-11-16,Beau Villa Agadir,agadir,Centre,villa,vente,1839131.22,264.15,2,2.0,0,2005
1068,ANO001259,2023-12-19,Duplex moderne Tétouan,tetouan,M'diq,duplex,location,2949.44,173.58,2,2.0,0,1995
743,ANO001176,2023-03-02,Appartement 80m² Talborjt,agadir,Talborjt,appartement,location,409134.40,80.05,1,1.0,0,2024
1279,ANO001013,2024-11-17,Beau Villa Tétouan,tetouan,Centre,villa,vente,2028725.40,263.64,3,3.0,0,2002
259,ANO001474,2024-08-20,Beau Appartement Meknès,meknes,Ville Nouvelle,appartement,location,283798.27,119.24,1,1.0,1,2001


In [107]:
df[df["prix"] <= 0]

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction


In [108]:
df[df["surface"] <= 0]

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction


In [110]:
df[df["nb_chambres"] < 0]

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction


In [111]:
df.info()

<class 'pandas.DataFrame'>
Index: 1500 entries, 0 to 1507
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   annonce_id          1500 non-null   str           
 1   date_publication    1500 non-null   datetime64[us]
 2   titre               1500 non-null   str           
 3   ville               1500 non-null   str           
 4   quartier            1500 non-null   str           
 5   type_bien           1500 non-null   category      
 6   transaction         1500 non-null   category      
 7   prix                1500 non-null   float64       
 8   surface             1500 non-null   float64       
 9   nb_chambres         1500 non-null   int64         
 10  nb_salles_bain      1500 non-null   float64       
 11  etage               1500 non-null   int64         
 12  annee_construction  1500 non-null   int64         
dtypes: category(2), datetime64[us](1), float64(3), int64(3), str(4)


In [112]:
df[[
    "prix",
    "surface",
    "nb_chambres",
    "nb_salles_bain",
    "etage"
]].describe()

,prix,surface,nb_chambres,nb_salles_bain,etage
count,1.500000e+03,1500.000000,1500.000000,1500.000000,1500.000000
mean,1.732155e+06,175.616767,1.522667,1.523000,1.870667
std,6.251217e+06,264.690740,1.337664,1.337846,2.335003
min,1.200000e+02,8.000000,0.000000,0.000000,0.000000
25%,1.154916e+04,79.882500,1.000000,1.000000,0.000000
50%,6.057739e+05,109.715000,1.000000,1.000000,1.000000
75%,1.821021e+06,217.300000,2.000000,2.000000,3.000000
max,8.000000e+07,3000.000000,13.000000,13.000000,10.000000


In [117]:
df[df["prix"] == df["prix"].min()]["prix"].value_counts()

prix
120.0    3
Name: count, dtype: int64

In [115]:
df[df["prix"] == 120]

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction
215,ANO000243,2023-02-25,Bureau 133m² Centre Ville,tanger,Centre Ville,bureau,location,120.0,133.90,0,0.0,5,2005
1169,ANO000014,2023-09-02,Appartement 103m² Centre,agadir,Centre,appartement,location,120.0,103.46,2,2.0,6,2011
1365,ANO001334,2023-10-02,Villa moderne Kenitra,kenitra,Centre,villa,location,120.0,180.92,3,3.0,0,2009


In [118]:
num_cols = ["prix", "surface", "nb_chambres", "nb_salles_bain", "etage"]

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    df[col + "_outlier"] = (
        (df[col] < Q1 - 1.5 * IQR) |
        (df[col] > Q3 + 1.5 * IQR)
    )

In [119]:
df["logic_anomaly"] = (
    ((df["nb_salles_bain"] == 0) & (df["surface"] > 30)) |
    ((df["nb_chambres"] == 0) & (df["surface"] > 40))
)

In [120]:
p99 = df["prix"].quantile(0.99)

df["luxury"] = df["prix"] > p99

In [121]:
df["suspicious_price"] = df["prix"] < 5000
df["suspicious_surface"] = df["surface"] < 15

In [122]:
df["is_anomaly"] = (
    df["prix_outlier"] |
    df["surface_outlier"] |
    df["nb_chambres_outlier"] |
    df["nb_salles_bain_outlier"] |
    df["logic_anomaly"] |
    df["suspicious_price"] |
    df["suspicious_surface"]
)

features

In [123]:
df["surface"].min()

np.float64(8.0)

In [124]:
df["prix"].min()

np.float64(120.0)

In [147]:
df["prix_par_m2"] = (df["prix"] / df["surface"]).round(2)

In [126]:
df["annee_construction"].min(), df["annee_construction"].max()

(np.int64(1961), np.int64(2024))

In [127]:
import datetime

current_year = datetime.datetime.now().year

df["age_estime"] = current_year - df["annee_construction"]

In [128]:
df["categorie_surface"] = pd.cut(
    df["surface"],
    bins=[0, 80, 150, float("inf")],
    labels=["petit", "moyen", "grand"]
)

In [129]:
df["year"] = df["date_publication"].dt.year

In [137]:
df["month"] = df["date_publication"].dt.strftime("%B")

In [131]:
df["quarter"] = df["date_publication"].dt.quarter

In [132]:
q1 = df["prix"].quantile(0.25)
q2 = df["prix"].quantile(0.50)
q3 = df["prix"].quantile(0.75)

In [133]:
df["categorie_prix"] = pd.cut(
    df["prix"],
    bins=[0, q1, q2, q3, df["prix"].max()],
    labels=["economique", "moyen", "haut_standing", "luxe"]
)

In [134]:
df["categorie_prix"].value_counts()

categorie_prix
economique       375
moyen            375
haut_standing    375
luxe             375
Name: count, dtype: int64

In [139]:
df.dtypes

annonce_id                           str
date_publication          datetime64[us]
titre                                str
ville                                str
quartier                             str
type_bien                       category
transaction                     category
prix                             float64
surface                          float64
nb_chambres                        int64
nb_salles_bain                   float64
etage                              int64
annee_construction                 int64
prix_outlier                        bool
surface_outlier                     bool
nb_chambres_outlier                 bool
nb_salles_bain_outlier              bool
etage_outlier                       bool
logic_anomaly                       bool
luxury                              bool
suspicious_price                    bool
suspicious_surface                  bool
is_anomaly                          bool
prix_par_m2                      float64
age_estime      

In [141]:
df["quarter"] = "Q" + df["date_publication"].dt.quarter.astype(str)

In [142]:
df["ville"] = df["ville"].astype("category")
df["quartier"] = df["quartier"].astype("category")
df["type_bien"] = df["type_bien"].astype("category")
df["transaction"] = df["transaction"].astype("category")

In [145]:
df.dtypes

annonce_id                           str
date_publication          datetime64[us]
titre                                str
ville                           category
quartier                        category
type_bien                       category
transaction                     category
prix                             float64
surface                          float64
nb_chambres                        int64
nb_salles_bain                   float64
etage                              int64
annee_construction                 int64
prix_outlier                        bool
surface_outlier                     bool
nb_chambres_outlier                 bool
nb_salles_bain_outlier              bool
etage_outlier                       bool
logic_anomaly                       bool
luxury                              bool
suspicious_price                    bool
suspicious_surface                  bool
is_anomaly                          bool
prix_par_m2                      float64
age_estime      

In [148]:
df.to_csv("data_clean.csv", index=False)